In [1]:
# Importar bibliotecas necesarias
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import warnings


In [2]:
# 1. Cargar los datos
df = pd.read_csv('/datasets/users_behavior.csv')

In [3]:
# 2. Separar características (X) y objetivo (y)
X = df.drop(columns='is_ultra')
y = df['is_ultra']

In [4]:
# 3. Dividir los datos: entrenamiento (60%), validación (20%), prueba (20%)
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full, test_size=0.25, random_state=42, stratify=y_train_full)


In [5]:
# 4. Escalar características
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)


In [6]:
# 5. Entrenar y evaluar modelos

# -- Modelo 1: Regresión logística
log_model = LogisticRegression(random_state=42)
log_model.fit(X_train_scaled, y_train)
log_pred = log_model.predict(X_valid_scaled)
log_acc = accuracy_score(y_valid, log_pred)

# --- Modelo 2: Árbol de decisión (buscando el mejor max_depth)
best_tree_acc = 0
best_tree_depth = 0
for depth in range(1, 20):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)  # Árboles no necesitan datos escalados
    pred = tree.predict(X_valid)
    acc = accuracy_score(y_valid, pred)
    if acc > best_tree_acc:
        best_tree_acc = acc
        best_tree_depth = depth

# Entrenar el mejor árbol
tree_model = DecisionTreeClassifier(max_depth=best_tree_depth, random_state=42)
tree_model.fit(X_train, y_train)

# --- Modelo 3: Random Forest (ajustando n_estimators y max_depth)
best_forest_acc = 0
best_forest_params = (0, 0)
for n in [10, 50, 100]:
    for depth in [3, 5, 10, None]:
        forest = RandomForestClassifier(n_estimators=n, max_depth=depth, random_state=42)
        forest.fit(X_train, y_train)
        pred = forest.predict(X_valid)
        acc = accuracy_score(y_valid, pred)
        if acc > best_forest_acc:
            best_forest_acc = acc
            best_forest_params = (n, depth)

# Entrenar el mejor modelo de Random Forest
best_n, best_depth = best_forest_params
forest_model = RandomForestClassifier(n_estimators=best_n, max_depth=best_depth, random_state=42)
forest_model.fit(X_train, y_train)



RandomForestClassifier(max_depth=10, n_estimators=10, random_state=42)

In [8]:
# 6. Comparar modelos
print("Resultados en el conjunto de validación:")
print(f"Regresión logística: {log_acc:.3f}")
print(f"Árbol de decisión (max_depth = {best_tree_depth}): {best_tree_acc:.3f}")
print(f"Random Forest (n = {best_n}, max_depth = {best_depth}): {best_forest_acc:.3f}")

# 7. Probar en el conjunto de prueba
final_model = forest_model  # Elegimos el mejor modelo
test_pred = final_model.predict(X_test)
test_acc = accuracy_score(y_test, test_pred)
print(f"\nPrecisión final en el conjunto de prueba: {test_acc:.3f}")

# 8. Prueba: predecir con datos al azar
random_preds = np.random.randint(0, 2, size=len(y_test))
random_acc = accuracy_score(y_test, random_preds)
print(f"\nPrueba de cordura (modelo aleatorio): {random_acc:.3f}")

Resultados en el conjunto de validación:
Regresión logística: 0.745
Árbol de decisión (max_depth = 5): 0.790
Random Forest (n = 10, max_depth = 10): 0.790

Precisión final en el conjunto de prueba: 0.812

Prueba de cordura (modelo aleatorio): 0.521


# Conclusión
Tras analizar los datos de comportamiento de los usuarios y entrenar distintos modelos de clasificación, se logró desarrollar un modelo predictivo capaz de recomendar con buena precisión si un cliente debería optar por el plan Smart o Ultra.
1. El modelo más efectivo fue Random Forest, gracias a su capacidad para manejar relaciones no lineales y reducir el sobreajuste.

2. Los atributos más determinantes para predecir el plan parecen estar relacionados con el uso de llamadas, mensajes y datos móviles.

3. El modelo es lo suficientemente confiable como para ser integrado en una herramienta de recomendación para nuevos usuarios.

Este modelo puede ser una valiosa herramienta para Megaline, ya que permite identificar patrones en el comportamiento de los usuarios y recomendarles el plan más adecuado, ayudando a migrar eficientemente a los clientes desde planes antiguos hacia las opciones actuales, mejorando la satisfacción del cliente y la rentabilidad de la empresa.